# ⚡ Opus2Short AI Studio — 1-Clique no Google Colab (Grátis + Google Drive)

Este notebook sobe o **Opus2Short AI Studio** (clone full-stack do OpusClip + 2Short.ai) gratuitamente no Google Colab, salvando seus projetos, clipes, Brand Kits e API Keys no seu **Google Drive** para nunca perder dados entre sessões.

### Como usar:
1. *(Opcional)* Vá em **Ambiente de execução → Alterar tipo de ambiente de execução** e escolha **T4 GPU** (mais rápido) ou **CPU** (mais horas contínuas usando Groq Whisper API).
2. Clique no botão **▶ (Executar)** na célula abaixo e autorize o acesso ao seu Google Drive.
3. Aguarde ~2 minutos e abra o link **`https://...trycloudflare.com`** que aparecer no final (funciona no notebook e no celular!).

In [ ]:
# @title 1. Conectar Google Drive (Salva seus projetos e API Keys para sempre)
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/Opus2ShortStudio'
os.makedirs(f'{DRIVE_BASE}/data', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/storage/uploads', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/storage/projects', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/storage/exports', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/storage/brandkits', exist_ok=True)
print('✅ Google Drive montado em:', DRIVE_BASE)

In [ ]:
# @title 2. Instalar dependências (Node 22, FFmpeg, yt-dlp, faster-whisper, OpenCV, cloudflared) e iniciar o Studio
import os, subprocess, time, re

REPO_DIR = '/content/rea-opusclip-and-2short'
DRIVE_BASE = '/content/drive/MyDrive/Opus2ShortStudio'

print('[1/5] Instalando Node.js 22, FFmpeg e fontes...')
!apt-get update -qq && apt-get install -y -qq ffmpeg libass-dev fonts-dejavu-core fonts-liberation curl >/dev/null
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null 2>&1 && apt-get install -y -qq nodejs >/dev/null
!node -v && ffmpeg -version | head -n 1

print('[2/5] Instalando yt-dlp, faster-whisper e OpenCV...')
!pip install -q yt-dlp faster-whisper opencv-python-headless

print('[3/5] Baixando/atualizando repositório Opus2Short Studio...')
if os.path.exists(f'{REPO_DIR}/.git'):
    !git -C {REPO_DIR} pull origin main
else:
    !git clone https://github.com/LANOSGITUFC/rea-opusclip-and-2short.git {REPO_DIR}

# Conectar pastas persistentes do Google Drive
!rm -rf {REPO_DIR}/app/data {REPO_DIR}/app/storage
!ln -s {DRIVE_BASE}/data {REPO_DIR}/app/data
!ln -s {DRIVE_BASE}/storage {REPO_DIR}/app/storage

print('[4/5] Compilando Frontend React e instalando Backend...')
!npm --prefix {REPO_DIR}/app/web install --no-audit --no-fund >/dev/null
!npm --prefix {REPO_DIR}/app/web run build
!npm --prefix {REPO_DIR}/app/server install --omit=dev --no-audit --no-fund >/dev/null

print('[5/5] Baixando cloudflared e iniciando servidor na porta 3000...')
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Encerrar instancias anteriores se houver
!pkill -f 'app/server/index.js' 2>/dev/null || true
!pkill -f 'cloudflared tunnel' 2>/dev/null || true

server_proc = subprocess.Popen(
    ['node', f'{REPO_DIR}/app/server/index.js'],
    cwd=f'{REPO_DIR}/app/server',
    stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT
)
time.sleep(2)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3000'],
    stdout=open('/content/tunnel.log', 'w'),
    stderr=subprocess.STDOUT
)

public_url = None
for _ in range(20):
    time.sleep(1)
    with open('/content/tunnel.log', 'r', errors='ignore') as f:
        m = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', f.read())
        if m:
            public_url = m.group(0)
            break

print('\n' + '='*60)
if public_url:
    print('🚀 OPUS2SHORT AI STUDIO NO AR! ABRA NO NOTEBOOK OU CELULAR:')
    print(f'👉 {public_url}')
else:
    print('⚠️ Não foi possível ler a URL. Veja /content/tunnel.log')
print('='*60)
print('Mantenha esta aba aberta enquanto estiver usando o Studio.')